# Importing modules and settings

### Importing libraries

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
import seaborn as sns
#import scrublet as scr

In [ ]:
from scipy.stats import kruskal

General settings of Scanpy

In [ ]:
sc.settings.verbosity = 3 
sc.logging.print_header()
sc.settings.set_figure_params(dpi=80, facecolor='white')


In [ ]:
umap_cmap = sns.blend_palette(['xkcd:light grey', 'xkcd:indigo'], as_cmap = True)

# Declaring the input and output files

In [ ]:
# Reading the Ingest results file
adata = sc.read_h5ad('./Smed_L78-L47_20250523_Ingest.h5ad')

In [ ]:
adata

In [ ]:
# Add the names of each Leiden resolution to a list
leiden_names = adata.obs.columns[adata.obs.columns.str.contains('leiden')].to_list()

In [ ]:
leiden_names = ['leiden_1',
 'leiden_1.5',
 'leiden_2',
 'leiden_2.5',
 'leiden_3']

In [ ]:
adata.var

In [ ]:
adata.obs.columns


# Declaring samples, clustering layer and ingest column

In [ ]:
samp = 'Sample'

In [ ]:
# Select the best Leiden resolution for your dataset
clusteringlayer = 'leiden_2.5'

In [ ]:
ref1 = 'sero'
ref2 = 'sizes'

In [ ]:
ingest1 = clusteringlayer + '_ingested_' + ref1
ingest2 = clusteringlayer + '_ingested_' + ref2

# Pandas dataframe with markers

In [ ]:
list(adata.uns)

In [ ]:
# Upload the Wilcox markers into a df
markers_w = pd.DataFrame(adata.uns['rank_genes_groups_wilcox_'+clusteringlayer]['names']).head(20)

In [ ]:
markers_w

In [ ]:
# Extract top genes based on p value
markers_w_l = pd.DataFrame(adata.uns['rank_genes_groups_wilcox_'+clusteringlayer]['pvals_adj']).head(20)

In [ ]:
# Upload the logreg markers into a df
markers_l = pd.DataFrame(adata.uns['rank_genes_groups_logreg_'+clusteringlayer]['names']).head(20)

In [ ]:
markers_l

# General Plots

In [ ]:
# Plot UMAP of the chosen Leiden resolution
with plt.rc_context({'figure.figsize': (12, 12)}):
    sc.pl.umap(adata, color=clusteringlayer, legend_loc='on data', legend_fontoutline = 5, title= 'Clustering layer '+str(clusteringlayer), size = 30,
        frameon=False)

In [ ]:
# Access and retrieve the categories of the clusteringlayer + '_ingested' column.
adata.obs[clusteringlayer + '_ingested_' + ref1].cat.categories

In [ ]:
# Access and retrieve the categories of the clusteringlayer + '_ingested' column.
adata.obs[clusteringlayer + '_ingested_' + ref2].cat.categories

In [ ]:
# Plot UMAP of Ingest Leiden with legend to the right
with plt.rc_context({'figure.figsize': (12, 12)}):
    sc.pl.umap(adata, color=ingest1, legend_fontoutline = 5, title= 'Clustering layer '+str(clusteringlayer) + ' ' + ref1, size = 30,
        frameon=False)

In [ ]:
# Legend to the right
with plt.rc_context({'figure.figsize': (12, 12)}):
    sc.pl.umap(adata, color= ingest2, legend_fontoutline = 5, title= 'Clustering layer '+str(clusteringlayer)+ ' ' + ref2, size = 30,
        frameon=False)

In [ ]:
# Iterate over each category of the categorical variable samp and creates a separate UMAP plot for each category
for r in adata.obs[samp].cat.categories:
    with plt.rc_context({'figure.figsize': (12, 12)}):
        sc.pl.umap(adata, color= samp, groups = r, legend_fontoutline = 5, title= samp + ': ' + r , size = 30,
            frameon=False)

In [ ]:
# Violin plot for number of unique genes
with plt.rc_context({'figure.figsize': (15, 5)}):
    sc.pl.violin(adata, keys = "n_genes" , groupby = clusteringlayer, jitter = False, rotation = 90)

In [ ]:
# Violin plot for number of times a gene is expressed (transcribed into RNA)
with plt.rc_context({'figure.figsize': (15, 5)}):
    sc.pl.violin(adata, keys = "n_counts" , groupby = clusteringlayer, jitter = False, rotation = 90)

In [ ]:
# Violin plot of the total number of RNA molecules detected within a cell or group of cells, irrespective of their gene identity
with plt.rc_context({'figure.figsize': (15, 5)}):
    sc.pl.violin(adata, keys = "total_counts" , groupby = clusteringlayer, jitter = False, rotation = 90)

In [ ]:
# Comparison between the different replicates cdh1 VS GFP VS H2B
sc.pl.violin(adata, keys = ['n_genes', 'total_counts', 'n_counts'] , groupby = samp, log = True, jitter = False, multi_panel = True, rotation = 90)

In [ ]:
# Comparison between the different sublibraries 
sc.pl.violin(adata, keys = ['n_genes', 'total_counts', 'n_counts'] , groupby = 'Library', log = True, jitter = False, multi_panel = True, rotation = 90)

In [ ]:
# Dot plot visualisation of the top differentially expressed genes across different clusters of cells
sc.pl.rank_genes_groups_dotplot(adata, n_genes=2, key = 'rank_genes_groups_wilcox_'+clusteringlayer, cmap = umap_cmap)

In [ ]:
# Dot plot visualisation of the top differentially expressed genes across different clusters of cells
sc.pl.rank_genes_groups_dotplot(adata, n_genes=2, key = 'rank_genes_groups_logreg_'+clusteringlayer, cmap = umap_cmap)

In [ ]:
# Matrix plot to visualize the top 4 marker genes for each cluster
sc.pl.rank_genes_groups_matrixplot(adata, n_genes=4, key = 'rank_genes_groups_wilcox_'+clusteringlayer)

In [ ]:
# Matrix plot to visualize the top 4 marker genes for each cluster
sc.pl.rank_genes_groups_matrixplot(adata, n_genes=4, key = 'rank_genes_groups_logreg_'+clusteringlayer)

In [ ]:
# Dendogram of the orgiginal Leiden clusters
with plt.rc_context({'figure.figsize': (15, 5)}):
    sc.pl.dendrogram(adata, groupby = clusteringlayer)

ingest2

In [ ]:
# Dendogram of the Ingested clusters
with plt.rc_context({'figure.figsize': (15, 5)}):
    sc.pl.dendrogram(adata, groupby = ingest1 )

In [ ]:
# Dendogram of the Ingested clusters
with plt.rc_context({'figure.figsize': (15, 5)}):
    sc.pl.dendrogram(adata, groupby = ingest2 )

In [ ]:
# UMAP of the number of counts (n_counts) per cell
with plt.rc_context({'figure.figsize': (12, 12)}):
    sc.pl.umap(adata, color='n_counts', legend_loc='on data', legend_fontoutline = 5, title= 'n counts', size = 30,
        frameon=False, add_outline = True)

In [ ]:
# UMAP of the number of genes (n_genes) per cell
with plt.rc_context({'figure.figsize': (12, 12)}):
    sc.pl.umap(adata, color='n_genes', legend_loc='on data', legend_fontoutline = 5, title= 'n genes', size = 30,
        frameon=False, add_outline = True)

In [ ]:
# UMAP of the n_genes_by_counts
with plt.rc_context({'figure.figsize': (12, 12)}):
    sc.pl.umap(adata, color='n_genes_by_counts', legend_loc='on data', legend_fontoutline = 5, title= 'n_genes_by_counts', size = 30,
        frameon=False, add_outline = True)

In [ ]:
# UMAP of the doublet_score
with plt.rc_context({'figure.figsize': (12, 12)}):
    sc.pl.umap(adata, color='doublet_score', legend_loc='on data', legend_fontoutline = 5, title= 'doublet_score', size = 30,
        frameon=False, add_outline = True)

In [ ]:
# UMAP of the neoblast_score
with plt.rc_context({'figure.figsize': (12, 12)}):
    sc.pl.umap(adata, color='neoblast_score', legend_loc='on data', legend_fontoutline = 5, title= 'neoblast_score', size = 30,
        frameon=False, add_outline = True)

# Stem cell scores

In [ ]:
adata.uns.keys()

In [ ]:
# retrieve the dataframe with teh percentages of GFP cells and H2B cells
percs_gfp_h2b = adata.uns['GFP_H2B_perc_leiden_2.5']

In [ ]:
percs_gfp_h2b

In [ ]:
# Bar plot of the ratios
data = pd.DataFrame((percs_gfp_h2b['H2B'] / percs_gfp_h2b['GFP']).sort_values(), columns=['ratio'])

with plt.rc_context({'figure.figsize': (20, 5)}):
    sns.barplot(x=data.index.tolist(), y='ratio', data= data)
    plt.xticks(rotation=90)
    plt.title('Ratio of H2B cells vs GFP control cells per cluster ( %GFP / % H2B)')

In [ ]:
# retrieve the neoblast score
neoblast_score = pd.DataFrame.from_dict(adata.uns['neoblast_score_leiden_2.5'], orient='index', columns=['neoblast_score'])
neoblast_score.index.name = 'leiden_2.5'

In [ ]:
# Bar plot of the neoblast score
data = neoblast_score.sort_values('neoblast_score', ascending = False)
with plt.rc_context({'figure.figsize': (20, 5)}):
    sns.barplot(x=data.index.tolist(), y='neoblast_score', data= data)
    plt.xticks(rotation=90)
    plt.title('neoblast score per cluster')

# Examining one cluster

#### The following section is used to examine the clusters one by one

In [ ]:
# UMAP of the Ingest Leiden clusters
with plt.rc_context({'figure.figsize': (12, 12)}):
    sc.pl.umap(adata, color=clusteringlayer, legend_loc='on data', legend_fontoutline = 5, title= 'Clustering layer '+str(clusteringlayer), size = 30,
        frameon=False, add_outline = True)

In [ ]:
# Select a cluster of interest
cl = '25'

In [ ]:
# Highlight specific cluster
with plt.rc_context({'figure.figsize': (8, 8)}):
    sc.pl.umap(adata, color = clusteringlayer, groups = cl, size = 15)

In [ ]:
# Scatter plot of the selected cluster
sc.pl.scatter(adata, x = 'n_counts', y = 'n_genes', color = clusteringlayer, groups = cl, size = 10)

In [ ]:
# Calculate the percentage of cells in specified clusters for each sample group
percs = adata[adata.obs[clusteringlayer] == cl].obs[samp].value_counts()/ adata.obs[samp].value_counts() * 100

# Bar plot from the calculated percentages
sns.barplot(data = percs, palette = list(adata.uns['Sample_colors']))
plt.xticks(rotation=90)
plt.ylabel('percentage')
plt.xlabel(samp)
plt.title('cluster: ' + cl)
plt.show()

In [ ]:
percs

In [ ]:
# Ingest label of the cluster (sero)
adata.obs.loc[adata.obs[clusteringlayer] == cl, clusteringlayer + '_ingested_' +  ref1][0]

In [ ]:
# Other labels (sero)
adata.obs.loc[adata.obs[clusteringlayer] == cl, ingest1].value_counts()

In [ ]:
# Bar plot of the distribution of values in the ingest column for cells that belong to the specific cluster (sero)
with plt.rc_context({'figure.figsize': (20, 3)}):
    sns.barplot(adata.obs.loc[adata.obs[clusteringlayer] == cl, ingest1].value_counts().sort_index(), palette = 'pastel')
    plt.xticks(rotation=90)

In [ ]:
# Ingest label of the cluster (sizes)
adata.obs.loc[adata.obs[clusteringlayer] == cl, clusteringlayer + '_ingested_' +  ref2][0]

In [ ]:
# Other labels (sizes)
adata.obs.loc[adata.obs[clusteringlayer] == cl, ingest2].value_counts()

In [ ]:
# Bar plot of the distribution of values in the ingest column for cells that belong to the specific cluster (sizes)
with plt.rc_context({'figure.figsize': (20, 3)}):
    sns.barplot(adata.obs.loc[adata.obs[clusteringlayer] == cl, ingest2].value_counts().sort_index(), palette = 'pastel')
    plt.xticks(rotation=90)

In [ ]:
top = 20

In [ ]:
adata.raw.var.columns

In [ ]:
# Visualise the markers with the genes based on wilxon method
adata.raw.var.loc[markers_w[cl].to_list()][['gene_type','gene_ddv6', 'gene_JakkeGuo', 'gene_Jakke_ver1', 'Preferred_name','Description.x']]

In [ ]:
markers_w[cl]

In [ ]:
# Extract the top marker genes for a specified cluster from a df
clmarkers = markers_w[cl].loc[0:(top-1)].to_list()

In [ ]:
# UMAP of these genes, visualising the expression patterns of these markers across all cells
sc.pl.umap(adata, color=clmarkers, cmap = umap_cmap)

In [ ]:
# Dot plot to visualise the expression of the top marker genes across different clusters
sc.pl.dotplot(adata, clmarkers, groupby= clusteringlayer, swap_axes = True, cmap = umap_cmap)

In [ ]:
markers_l[cl]

In [ ]:
# Visualise the markers with the genes based on log reg method
adata.raw.var.loc[markers_l[cl].to_list()][['gene_type','gene_ddv6', 'gene_JakkeGuo', 'gene_Jakke_ver1', 'Preferred_name','Description.x']]

In [ ]:
# UMAP of these genes, visualising the expression patterns of these markers across all cells
clmarkersl = markers_l[cl].loc[0:(top-1)].to_list()
sc.pl.umap(adata, color=clmarkersl, cmap = umap_cmap)

In [ ]:
# Dot plot to visualise the expression of the top marker genes across different clusters
sc.pl.dotplot(adata, clmarkersl, groupby= clusteringlayer, swap_axes = True, cmap = umap_cmap)

### Top Marker Gene (LogReg Method)

In [ ]:
# Extract top-ranked gene for LogReg
top_marker_gene_logreg = markers_l[cl].iloc[0]

In [ ]:
cluster_cells = adata[adata.obs[clusteringlayer] == cl]

In [ ]:
cluster_cells

In [ ]:
# Mean expression for LogReg marker
mean_exp_logreg = cluster_cells[:, top_marker_gene_logreg].X.mean()  
percentage_logreg = (cluster_cells[:, top_marker_gene_logreg].X > 0).mean() * 100

In [ ]:
# Print the top marker gene 
print(f"Top marker gene by LogReg in cluster {cl} is: {top_marker_gene_logreg}, expressed in {percentage_logreg:.2f}% of cells")

### Top Marker Gene (Wilcoxon Method)

In [ ]:
# Extract top-ranked gene for Wilcoxon
top_marker_gene_wilcoxon = markers_w[cl].iloc[0]

In [ ]:
mean_exp_wilcoxon = cluster_cells[:, top_marker_gene_wilcoxon].X.mean()  # Mean expression for Wilcoxon marker
percentage_wilcoxon = (cluster_cells[:, top_marker_gene_wilcoxon].X > 0).mean() * 100

In [ ]:
print(f"Top marker gene by Wilcoxon in cluster {cl} is: {top_marker_gene_wilcoxon}, expressed in {percentage_wilcoxon:.2f}% of cells")

### Printing All Statements

In [ ]:
# Print all of the statements together
print(f"Top marker gene by LogReg in cluster {cl} is: {top_marker_gene_logreg}, expressed in {percentage_logreg:.2f}% of cells")
print(f"Top marker gene by Wilcoxon in cluster {cl} is: {top_marker_gene_wilcoxon}, expressed in {percentage_wilcoxon:.2f}% of cells")

In [ ]:
#Plot Most Expressed Gene, Top LogReg Marker and Top Wilcoxon Marker for a Cluster

plt.clf()  
plt.close()

# Compare the genes
genes = [top_marker_gene_logreg, top_marker_gene_wilcoxon]
unique_genes = list(set(genes))  # Get unique genes

# Create UMAP plots for all unique genes
for gene in unique_genes:
    with plt.rc_context({'figure.figsize': (8, 7)}):  # Set figure size context
        sc.pl.umap(
            adata,
            color=[gene],  # Use a list format for compatibility with scanpy
            cmap=umap_cmap,  # Assuming umap_cmap is defined earlier in your script
            title=f"Expression of {gene}",  # Dynamic title
            size=20,  # Adjust this to make the dots more defined
            show=True  # Display the plot; replace with save if you want to save instead
        )


check expression of markers

In [ ]:
gene = 'h1SMcG0012908'

In [ ]:
 with plt.rc_context({'figure.figsize': (8, 7)}):  # Set figure size context
        sc.pl.umap(
            adata,
            color=[gene],  # Use a list format for compatibility with scanpy
            cmap=umap_cmap,  # Assuming umap_cmap is defined earlier in your script
            title=f"Expression of {gene}",  # Dynamic title
            size=20,  # Adjust this to make the dots more defined
            show=True  # Display the plot; replace with save if you want to save instead
        )


In [ ]:
gene = 'h1SMcG0013999'

In [ ]:
 with plt.rc_context({'figure.figsize': (8, 7)}):  # Set figure size context
        sc.pl.umap(
            adata,
            color=[gene],  # Use a list format for compatibility with scanpy
            cmap=umap_cmap,  # Assuming umap_cmap is defined earlier in your script
            title=f"Expression of {gene}",  # Dynamic title
            size=20,  # Adjust this to make the dots more defined
            show=True  # Display the plot; replace with save if you want to save instead
        )


2 clusters at the same time

In [ ]:
cl = ['53', '82'] 

In [ ]:
# Highlight specific cluster
with plt.rc_context({'figure.figsize': (8, 8)}): 
    sc.pl.umap(adata, color = clusteringlayer, groups = cl, size = 15)

In [ ]:
# number of common markers between the 2 clusters
# wilkoxon markers
li1 = markers_w[cl[0]].tolist()
li2 = markers_w[cl[1]].tolist()
#li3 = markers_w[cl[2]].tolist()
li_com = [i for i in li1 if i in li2]
#li_com = [i for i in li1 if i in li2 and i in li3]
len(li_com)

In [ ]:
# number of common markers between the 2 clusters
# logreg markers
li1 = markers_l[cl[0]].tolist()
li2 = markers_l[cl[1]].tolist()
#li3 = markers_l[cl[2]].tolist()
li_com = [i for i in li1 if i in li2]
#li_com = [i for i in li1 if i in li2 and i in li3]
len(li_com)